# LeetCode 870 — Advantage Shuffle bằng Python

Link đề: https://leetcode.com/problems/advantage-shuffle/ · Độ khó: **Medium** · Chủ đề: Greedy (tham lam), Array (mảng), Two Pointers (hai con trỏ), Sorting (sắp xếp)

Skeleton LeetCode:

```python
class Solution:
    def advantageCount(self, nums1: List[int], nums2: List[int]) -> List[int]:
        ...
```

Ghi chú: bài toán này giống **cuộc đua ngựa của Điền Kỵ** (Tian Ji): `nums2` là đội ngựa của đối thủ (đã xếp cố định theo thứ tự), `nums1` là đội ngựa của ta nhưng ta được **chọn con nào ra đấu với con nào**. Ta muốn thắng nhiều lượt nhất. Đề cho phép trả về **bất kỳ** hoán vị tối ưu, nên khi kiểm thử ta dùng **hàm kiểm tra hợp lệ** (kết quả là hoán vị của `nums1` và số lượt thắng bằng giá trị tối ưu) chứ không so từng phần tử với một đáp án cố định.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: sắp xếp lại `nums1` sao cho số chỉ số `i` có `nums1[i] > nums2[i]` (gọi là **lợi thế**) là lớn nhất, và hiểu vì sao đề chấp nhận **nhiều đáp án**.
2. Nắm các công cụ Python cần thiết: `sorted(..., key=...)` và `sorted((giá trị, chỉ số) ...)` để **nhớ vị trí gốc**, `bisect.bisect_right`, `collections.deque` (lấy hai đầu), con trỏ `i`/`j` trên mảng đã sắp xếp, `Counter` để kiểm tra hoán vị.
3. Hiểu nguyên lý **tham lam bằng lập luận hoán đổi (exchange argument)**: dùng con ngựa **nhỏ nhất đủ sức thắng**, và con ngựa không thể thắng ai thì **hy sinh** cho đối thủ mạnh nhất.
4. Đi qua 4 cách: thử mọi hoán vị, tham lam bằng danh sách đã sắp xếp + `bisect`, tham lam mạnh-nhất-trước bằng `deque`, và hai con trỏ trên hai mảng đã sắp xếp (cách nộp chính); đối chiếu bằng oracle vét cạn, oracle **ghép cặp cực đại (Kuhn)** và oracle đặc trưng "k lượt thắng khả thi ⇔ k ngựa mạnh nhất thắng k đối thủ yếu nhất".
5. Tránh các lỗi kinh điển: sắp xếp `nums2` mà làm mất vị trí gốc, tìm chỉ số bằng `index` khi có giá trị trùng, so sánh `>=` thay vì `>`, và tham lam sai (luôn dùng ngựa mạnh nhất).

## 2. Đề bài & ví dụ

Cho hai mảng số nguyên `nums1` và `nums2` **cùng độ dài**. **Lợi thế** của `nums1` đối với `nums2` là số chỉ số `i` sao cho `nums1[i] > nums2[i]`.

Trả về **một hoán vị bất kỳ** của `nums1` làm **tối đa hóa** lợi thế đối với `nums2`.

**Ví dụ 1**

- Input: `nums1 = [2,7,11,15], nums2 = [1,10,4,11]`
- Output: `[2,11,7,15]` (lợi thế `4`: `2>1`, `11>10`, `7>4`, `15>11`)

**Ví dụ 2**

- Input: `nums1 = [12,24,8,32], nums2 = [13,25,32,11]`
- Output: `[24,32,8,12]` (lợi thế `3`: `24>13`, `32>25`, `8<32` thua, `12>11`)

**Ràng buộc:**

- `1 <= nums1.length <= 10^5`
- `nums2.length == nums1.length`
- `0 <= nums1[i], nums2[i] <= 10^9`

Lưu ý: đề nói "trả về **bất kỳ** hoán vị nào tối đa hóa lợi thế". Với ví dụ 1 có nhiều đáp án khác cũng đạt lợi thế 4 (ví dụ `[2, 11, 7, 15]` là một; thử đổi chỗ để xem).

In [1]:
from typing import List
from itertools import permutations
from collections import Counter, deque
from bisect import bisect_right
import random, time

examples = [
    ([2, 7, 11, 15], [1, 10, 4, 11], [2, 11, 7, 15], 4),
    ([12, 24, 8, 32], [13, 25, 32, 11], [24, 32, 8, 12], 3),
]

def advantage(a: List[int], b: List[int]) -> int:
    """Số chỉ số i có a[i] > b[i]."""
    return sum(1 for x, y in zip(a, b) if x > y)

for a, b, out, adv in examples:
    print(f"nums1={a} nums2={b} -> {out}, lợi thế {advantage(out, b)} (kỳ vọng {adv})")
    assert advantage(out, b) == adv

nums1=[2, 7, 11, 15] nums2=[1, 10, 4, 11] -> [2, 11, 7, 15], lợi thế 4 (kỳ vọng 4)
nums1=[12, 24, 8, 32] nums2=[13, 25, 32, 11] -> [24, 32, 8, 12], lợi thế 3 (kỳ vọng 3)


## 3. Kiến thức nền

### 3.1. Vì sao có nhiều đáp án và cách kiểm tra

Có thể có nhiều hoán vị cùng đạt lợi thế tối đa. Vì vậy khi kiểm thử ta cần:

1. `Counter(kết quả) == Counter(nums1)` (đúng là hoán vị của `nums1`, kể cả giá trị lặp);
2. `advantage(kết quả, nums2) ==` giá trị tối ưu (do một oracle độc lập tính ra).

Hàm kiểm tra này **không so sánh với một đáp án cố định**.

In [2]:
a, b = [2, 7, 11, 15], [1, 10, 4, 11]
cands = [p for p in permutations(a) if advantage(p, b) == 4]
print("số hoán vị (theo vị trí) có lợi thế 4:", len(cands), "| ví dụ:", cands[:3])

số hoán vị (theo vị trí) có lợi thế 4: 1 | ví dụ: [(2, 11, 7, 15)]


### 3.2. Sắp xếp mà vẫn nhớ vị trí gốc

Ta cần sắp xếp `nums2` để xử lý theo thứ tự yếu -> mạnh, nhưng lúc trả kết quả phải đặt đúng **vị trí gốc**. Cách chuẩn: sắp xếp danh sách cặp `(giá trị, chỉ số gốc)` bằng `sorted((v, i) for i, v in enumerate(nums2))`. Cặp được so sánh theo giá trị trước, chỉ số sau (khi giá trị trùng nhau).

In [3]:
nums2 = [5, 1, 5, 3]
pairs = sorted((v, i) for i, v in enumerate(nums2))
print(pairs)                       # giá trị tăng dần, kèm chỉ số gốc
print([i for v, i in pairs])       # thứ tự chỉ số khi đi từ yếu tới mạnh

[(1, 1), (3, 3), (5, 0), (5, 2)]
[1, 3, 0, 2]


### 3.3. `bisect_right` và `deque`

- `bisect_right(s, x)`: trong danh sách đã sắp xếp `s`, trả về vị trí đầu tiên mà phần tử **lớn hơn `x`** (nghĩa là `s[k] > x`); nếu mọi phần tử `<= x` thì trả `len(s)`. Đây chính là "tìm con ngựa **nhỏ nhất thắng được** `x`".
- `deque`: danh sách hai đầu, `popleft()` / `pop()` đều `O(1)`: dùng để lấy con ngựa yếu nhất hoặc mạnh nhất.

In [4]:
s = [1, 3, 3, 7, 9]
for x in (0, 3, 8, 9):
    k = bisect_right(s, x)
    print(f"x={x}: bisect_right={k} -> {'con nhỏ nhất > x là ' + str(s[k]) if k < len(s) else 'không có con nào > x'}")

dq = deque([1, 3, 3, 7, 9])
print("yếu nhất:", dq.popleft(), "| mạnh nhất:", dq.pop(), "| còn lại:", list(dq))

x=0: bisect_right=0 -> con nhỏ nhất > x là 1
x=3: bisect_right=3 -> con nhỏ nhất > x là 7
x=8: bisect_right=4 -> con nhỏ nhất > x là 9
x=9: bisect_right=5 -> không có con nào > x
yếu nhất: 1 | mạnh nhất: 9 | còn lại: [3, 3, 7]


## 4. Ý tưởng nền tảng / trực giác (tính tay)

Hình dung cuộc đua ngựa: mỗi lượt ta chọn một con ngựa của mình ra đấu với con ngựa tương ứng của đối thủ; thắng khi **mạnh hơn hẳn** (`>`).

Với ví dụ `nums1 = [2,7,11,15]`, `nums2 = [1,10,4,11]`:

1. Đối thủ yếu nhất là `1`: ta dùng con **nhỏ nhất đủ sức thắng** nó là `2` (không phí `15` vào chỗ này).
2. Đối thủ kế `4`: con nhỏ nhất của ta còn lại thắng được là `7`.
3. Đối thủ kế `10`: con nhỏ nhất thắng được là `11`.
4. Đối thủ mạnh nhất `11`: con còn lại là `15` thắng được.

Cả 4 lượt thắng. Nếu ngược lại có một đối thủ quá mạnh mà **không con nào của ta thắng nổi**, ta nên **hy sinh** con **yếu nhất** của mình (nó chẳng thắng ai) cho đối thủ đó, để dành các con mạnh hơn cho các lượt có thể thắng.

## 5. Cách 1 — Brute Force: thử mọi hoán vị

**Ý tưởng.** Sinh mọi hoán vị của `nums1`, tính lợi thế của từng hoán vị, giữ hoán vị tốt nhất. Đây là định nghĩa trực tiếp của bài toán, nên cũng là **oracle** cho `n` nhỏ.

In [5]:
def advantage_brute(nums1: List[int], nums2: List[int]) -> List[int]:
    best, best_perm = -1, None
    for perm in permutations(nums1):
        cur = advantage(perm, nums2)
        if cur > best:
            best, best_perm = cur, list(perm)
    return best_perm

for a, b, out, adv in examples:
    res = advantage_brute(a, b)
    print(res, "lợi thế", advantage(res, b))
    assert Counter(res) == Counter(a) and advantage(res, b) == adv

[2, 11, 7, 15] lợi thế 4
[24, 32, 8, 12] lợi thế 3


**Độ phức tạp và nút thắt.** `O(n! * n)`: với `n = 10` đã là `~36 triệu` thao tác, còn `n = 10^5` là không thể. Hãy xem thời gian:

In [6]:
rng = random.Random(870)
for n in (5, 6, 7, 8, 9):
    a = [rng.randint(0, 20) for _ in range(n)]
    b = [rng.randint(0, 20) for _ in range(n)]
    t0 = time.perf_counter()
    advantage_brute(a, b)
    print(f"n={n}: {1000 * (time.perf_counter() - t0):8.1f} ms")

n=5:      0.1 ms
n=6:      0.8 ms
n=7:      4.6 ms
n=8:     41.5 ms


n=9:    323.0 ms


Thời gian tăng gần **gấp `n` lần mỗi khi thêm một phần tử**. Nút thắt: ta thử **mọi cách ghép** trong khi bài toán có cấu trúc **tham lam** rõ ràng (ngựa mạnh hơn luôn thắng được tập đối thủ không nhỏ hơn).

## 6. Ý tưởng thuật toán / bước đột phá

Đặt `a` là các con ngựa của ta, `b` là đối thủ. Hai nhận xét cốt lõi (sẽ chứng minh ở mục 10):

1. **Nếu thắng được một đối thủ, hãy dùng con ngựa nhỏ nhất đủ sức.** Con mạnh hơn dư sức ấy hữu ích hơn ở chỗ khác.
2. **Nếu một con ngựa chẳng thắng nổi ai (hay đối thủ quá mạnh), hãy hy sinh nó cho đối thủ mạnh nhất còn lại.** Một lượt thua là không tránh khỏi, nên hãy để lượt thua rơi vào ngựa yếu nhất và đối thủ khó thắng nhất.

Từ đó có ba cách cài đặt tham lam:

| Cách | Ý tưởng |
|---|---|
| 2 | Duyệt đối thủ theo thứ tự gốc; mỗi đối thủ lấy con **nhỏ nhất > đối thủ** từ danh sách đã sắp xếp (`bisect`), không có thì hy sinh con nhỏ nhất |
| 3 | Duyệt đối thủ từ **mạnh nhất** đến yếu nhất; nếu con mạnh nhất của ta thắng thì dùng, không thì hy sinh con yếu nhất (Điền Kỵ) |
| 4 | Duyệt ngựa của ta từ **yếu nhất**: thắng được đối thủ yếu nhất còn lại thì ghép, không thì hy sinh cho đối thủ mạnh nhất (hai con trỏ) |

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Danh sách đã sắp xếp + `bisect` (nhỏ nhất đủ sức thắng)

**Ý tưởng khác.** Sắp xếp `nums1` thành danh sách `s`. Với mỗi `y = nums2[i]` theo **thứ tự gốc**:

- `k = bisect_right(s, y)`: nếu `k < len(s)`, con nhỏ nhất thắng được là `s[k]`: lấy ra và gán cho vị trí `i`;
- nếu không (`k == len(s)`: không con nào thắng) thì lấy con **nhỏ nhất** `s[0]`.

Cách này không cần sắp xếp `nums2`, nhưng `s.pop(k)` giữa danh sách là `O(n)` nên tổng `O(n^2)` ở trường hợp xấu (vẫn nhanh với `n` vừa).

**Giải thích từng đoạn.**

- `s = sorted(nums1)`: ngựa của ta, tăng dần.
- `res = []`: kết quả theo thứ tự đối thủ gốc, nên không cần nhớ chỉ số.
- `bisect_right(s, y)`: phần tử đầu tiên `> y` (khớp đúng điều kiện thắng nghiêm ngặt).

In [7]:
def advantage_bisect(nums1: List[int], nums2: List[int]) -> List[int]:
    s = sorted(nums1)
    res = []
    for y in nums2:
        k = bisect_right(s, y)
        res.append(s.pop(k if k < len(s) else 0))
    return res

for a, b, out, adv in examples:
    res = advantage_bisect(a, b)
    print(res)
    assert Counter(res) == Counter(a) and advantage(res, b) == adv

[2, 11, 7, 15]
[24, 32, 8, 12]


**Code nộp LeetCode (Cách 2):**

In [8]:
class Solution:
    def advantageCount(self, nums1: List[int], nums2: List[int]) -> List[int]:
        from bisect import bisect_right
        s = sorted(nums1)
        res = []
        for y in nums2:
            k = bisect_right(s, y)
            res.append(s.pop(k if k < len(s) else 0))
        return res

sol_bisect = Solution().advantageCount
for a, b, out, adv in examples:
    r = sol_bisect(list(a), list(b))
    assert Counter(r) == Counter(a) and advantage(r, b) == adv
print("Cách 2 qua ví dụ")

Cách 2 qua ví dụ


**Độ phức tạp (Cách 2):** sắp xếp `O(n log n)`; mỗi bước `bisect` `O(log n)` và `pop` `O(n)` (dịch phần tử) ⇒ `O(n^2)` tệ nhất nhưng hằng số nhỏ (`memmove`); bộ nhớ `O(n)`.

---

### 7.2. Cách 3 — Duyệt đối thủ từ mạnh nhất, dùng `deque` (Điền Kỵ)

**Ý tưởng khác.** Sắp xếp `nums1` đưa vào `deque` `dq`. Sắp xếp các cặp `(giá trị, chỉ số)` của `nums2` và duyệt **từ mạnh nhất xuống yếu nhất**:

- nếu con **mạnh nhất** của ta `dq[-1] > y`: nó thắng, gán nó cho vị trí đối thủ này (`dq.pop()`);
- nếu không: không ai thắng được đối thủ này (vì ngay cả con mạnh nhất cũng thua), hy sinh con **yếu nhất** `dq.popleft()`.

Hai thao tác `pop` đều `O(1)` nên tổng `O(n log n)` chỉ để sắp xếp.

**Giải thích từng đoạn.**

- `order = sorted(range(n), key=lambda i: nums2[i])`: các chỉ số của đối thủ xếp theo giá trị tăng dần; `reversed(order)` đi từ mạnh nhất.
- `res[i]`: đặt đúng vị trí gốc của đối thủ.

In [9]:
def advantage_deque(nums1: List[int], nums2: List[int]) -> List[int]:
    n = len(nums1)
    dq = deque(sorted(nums1))
    res = [0] * n
    for i in sorted(range(n), key=lambda i: nums2[i], reverse=True):
        if dq[-1] > nums2[i]:
            res[i] = dq.pop()
        else:
            res[i] = dq.popleft()
    return res

for a, b, out, adv in examples:
    res = advantage_deque(a, b)
    print(res)
    assert Counter(res) == Counter(a) and advantage(res, b) == adv

[2, 11, 7, 15]
[24, 32, 8, 12]


**Code nộp LeetCode (Cách 3):**

In [10]:
class Solution:
    def advantageCount(self, nums1: List[int], nums2: List[int]) -> List[int]:
        from collections import deque
        n = len(nums1)
        dq = deque(sorted(nums1))
        res = [0] * n
        for i in sorted(range(n), key=lambda i: nums2[i], reverse=True):
            if dq[-1] > nums2[i]:
                res[i] = dq.pop()
            else:
                res[i] = dq.popleft()
        return res

sol_deque = Solution().advantageCount
for a, b, out, adv in examples:
    r = sol_deque(list(a), list(b))
    assert Counter(r) == Counter(a) and advantage(r, b) == adv
print("Cách 3 qua ví dụ")

Cách 3 qua ví dụ


**Độ phức tạp (Cách 3):** `O(n log n)` thời gian (sắp xếp), `O(n)` bộ nhớ.

---

### 7.3. Cách 4 — Hai con trỏ trên hai mảng đã sắp xếp (cách nộp chính)

**Ý tưởng khác.** Sắp xếp `nums1` tăng dần; sắp xếp các cặp `t = sorted((v, i) for i, v in enumerate(nums2))`. Dùng hai con trỏ trên `t`: `i` trỏ **đối thủ yếu nhất còn lại**, `j` trỏ **đối thủ mạnh nhất còn lại**. Duyệt từng con `v` của ta từ yếu đến mạnh:

- nếu `v > t[i][0]` (thắng được đối thủ yếu nhất còn lại): ghép `v` với đối thủ đó, `i += 1`;
- nếu không (`v` không thắng được ai còn lại, vì ngay đối thủ yếu nhất cũng không thắng): hy sinh cho đối thủ **mạnh nhất** `t[j]`, `j -= 1`.

**Giải thích từng đoạn.**

- `ans[t[i][1]] = v`: `t[i][1]` là chỉ số gốc của đối thủ, nên `v` đặt đúng vị trí.
- Mỗi con `v` được dùng **đúng một lần** và mỗi bước dịch `i` hoặc `j` đúng một đơn vị, nên tổng cộng gán đủ `n` vị trí.
- Con ngựa yếu nhất được xét trước: nó thắng được đối thủ yếu nhất thì dùng (đó là con **nhỏ nhất** đủ sức thắng đối thủ yếu nhất); nếu không thì nó chẳng thắng được ai nên hy sinh.

In [11]:
def advantage_two_pointers(nums1: List[int], nums2: List[int]) -> List[int]:
    a = sorted(nums1)
    t = sorted((v, i) for i, v in enumerate(nums2))
    n = len(nums2)
    ans = [0] * n
    i, j = 0, n - 1
    for v in a:
        if v > t[i][0]:
            ans[t[i][1]] = v
            i += 1
        else:
            ans[t[j][1]] = v
            j -= 1
    return ans

for a, b, out, adv in examples:
    res = advantage_two_pointers(a, b)
    print(res)
    assert Counter(res) == Counter(a) and advantage(res, b) == adv

[2, 11, 7, 15]
[24, 32, 8, 12]


**Code nộp LeetCode (Cách 4):**

In [12]:
class Solution:
    def advantageCount(self, nums1: List[int], nums2: List[int]) -> List[int]:
        nums1.sort()
        t = sorted((v, i) for i, v in enumerate(nums2))
        n = len(nums2)
        ans = [0] * n
        i, j = 0, n - 1
        for v in nums1:
            if v > t[i][0]:
                ans[t[i][1]] = v
                i += 1
            else:
                ans[t[j][1]] = v
                j -= 1
        return ans

sol_two = Solution().advantageCount
for a, b, out, adv in examples:
    r = sol_two(list(a), list(b))
    assert Counter(r) == Counter(a) and advantage(r, b) == adv
print("Cách 4 qua ví dụ")

Cách 4 qua ví dụ


**Độ phức tạp (Cách 4):** `O(n log n)` thời gian (hai lần sắp xếp), `O(n)` bộ nhớ (mảng `t` và `ans`).

## 8. TRACE từng bước

### 8.1. Cách 4 cho ví dụ 2: `nums1 = [12,24,8,32]`, `nums2 = [13,25,32,11]`

Sau sắp xếp: `a = [8, 12, 24, 32]`, `t = [(11,3), (13,0), (25,1), (32,2)]`. Con trỏ `i = 0`, `j = 3`.

| `v` | đối thủ yếu nhất `t[i]` | thắng? | hành động | `ans` |
|---|---|---|---|---|
| 8 | `(11,3)` | `8 > 11` sai | hy sinh cho `t[3] = (32,2)`: `ans[2] = 8`, `j = 2` | `[_, _, 8, _]` |
| 12 | `(11,3)` | `12 > 11` đúng | `ans[3] = 12`, `i = 1` | `[_, _, 8, 12]` |
| 24 | `(13,0)` | đúng | `ans[0] = 24`, `i = 2` | `[24, _, 8, 12]` |
| 32 | `(25,1)` | đúng | `ans[1] = 32`, `i = 3` | `[24, 32, 8, 12]` |

In [13]:
def trace_two(nums1, nums2):
    a = sorted(nums1)
    t = sorted((v, i) for i, v in enumerate(nums2))
    n = len(nums2)
    ans = [None] * n
    i, j = 0, n - 1
    print("a =", a, "| t =", t)
    for v in a:
        if v > t[i][0]:
            print(f"v={v:>3}: thắng đối thủ yếu nhất {t[i]} -> ans[{t[i][1]}]={v}")
            ans[t[i][1]] = v
            i += 1
        else:
            print(f"v={v:>3}: không thắng được ai (yếu nhất {t[i]}) -> hy sinh cho mạnh nhất {t[j]}: ans[{t[j][1]}]={v}")
            ans[t[j][1]] = v
            j -= 1
        print("        ans =", ans)
    return ans

res = trace_two([12, 24, 8, 32], [13, 25, 32, 11])
assert res == [24, 32, 8, 12]

a = [8, 12, 24, 32] | t = [(11, 3), (13, 0), (25, 1), (32, 2)]
v=  8: không thắng được ai (yếu nhất (11, 3)) -> hy sinh cho mạnh nhất (32, 2): ans[2]=8
        ans = [None, None, 8, None]
v= 12: thắng đối thủ yếu nhất (11, 3) -> ans[3]=12
        ans = [None, None, 8, 12]
v= 24: thắng đối thủ yếu nhất (13, 0) -> ans[0]=24
        ans = [24, None, 8, 12]
v= 32: thắng đối thủ yếu nhất (25, 1) -> ans[1]=32
        ans = [24, 32, 8, 12]


### 8.2. Cách 3 (Điền Kỵ) cho ví dụ 2

`dq = [8, 12, 24, 32]`. Đối thủ từ mạnh đến yếu: `32 (chỉ số 2)`, `25 (1)`, `13 (0)`, `11 (3)`.

- `32`: `dq[-1] = 32 > 32`? Không: hy sinh `8` (`popleft`), `res[2] = 8`.
- `25`: `dq[-1] = 32 > 25`: dùng `32`, `res[1] = 32`.
- `13`: `dq[-1] = 24 > 13`: dùng `24`, `res[0] = 24`.
- `11`: `dq[-1] = 12 > 11`: dùng `12`, `res[3] = 12`.

Kết quả `[24, 32, 8, 12]`: trùng với Cách 4.

In [14]:
def trace_deque(nums1, nums2):
    n = len(nums1)
    dq = deque(sorted(nums1))
    res = [None] * n
    for i in sorted(range(n), key=lambda i: nums2[i], reverse=True):
        y = nums2[i]
        if dq[-1] > y:
            res[i] = dq.pop()
            print(f"đối thủ {y:>3} (chỉ số {i}): mạnh nhất của ta thắng -> {res[i]}   còn {list(dq)}")
        else:
            res[i] = dq.popleft()
            print(f"đối thủ {y:>3} (chỉ số {i}): không ai thắng, hy sinh yếu nhất -> {res[i]}   còn {list(dq)}")
    return res

assert trace_deque([12, 24, 8, 32], [13, 25, 32, 11]) == [24, 32, 8, 12]

đối thủ  32 (chỉ số 2): không ai thắng, hy sinh yếu nhất -> 8   còn [12, 24, 32]
đối thủ  25 (chỉ số 1): mạnh nhất của ta thắng -> 32   còn [12, 24]
đối thủ  13 (chỉ số 0): mạnh nhất của ta thắng -> 24   còn [12]
đối thủ  11 (chỉ số 3): mạnh nhất của ta thắng -> 12   còn []


### 8.3. Cách 2 (`bisect`) cho ví dụ 1

`s = [2, 7, 11, 15]`. Duyệt `nums2` theo thứ tự gốc `[1, 10, 4, 11]`: `y=1` -> `bisect_right = 0` -> lấy `2`; `y=10` -> con nhỏ nhất `> 10` là `11`; `y=4` -> `7`; `y=11` -> `15`. Kết quả `[2, 11, 7, 15]`.

In [15]:
def trace_bisect(nums1, nums2):
    s = sorted(nums1)
    res = []
    for y in nums2:
        k = bisect_right(s, y)
        pick = k if k < len(s) else 0
        print(f"y={y:>3}: s={s} bisect_right={k} -> lấy s[{pick}]={s[pick]}")
        res.append(s.pop(pick))
    return res

assert trace_bisect([2, 7, 11, 15], [1, 10, 4, 11]) == [2, 11, 7, 15]

y=  1: s=[2, 7, 11, 15] bisect_right=0 -> lấy s[0]=2
y= 10: s=[7, 11, 15] bisect_right=1 -> lấy s[1]=11
y=  4: s=[7, 15] bisect_right=0 -> lấy s[0]=7
y= 11: s=[15] bisect_right=0 -> lấy s[0]=15


## 9. Code nộp LeetCode

Ta chọn **Cách 4** (hai con trỏ trên hai mảng đã sắp xếp): `O(n log n)` thời gian, ngắn gọn và không có thao tác `O(n)` ẩn như `list.pop(k)` ở Cách 2.

```python
class Solution:
    def advantageCount(self, nums1: List[int], nums2: List[int]) -> List[int]:
        nums1.sort()
        t = sorted((v, i) for i, v in enumerate(nums2))
        n = len(nums2)
        ans = [0] * n
        i, j = 0, n - 1
        for v in nums1:
            if v > t[i][0]:
                ans[t[i][1]] = v
                i += 1
            else:
                ans[t[j][1]] = v
                j -= 1
        return ans
```

In [16]:
class Solution:
    def advantageCount(self, nums1: List[int], nums2: List[int]) -> List[int]:
        nums1.sort()
        t = sorted((v, i) for i, v in enumerate(nums2))
        n = len(nums2)
        ans = [0] * n
        i, j = 0, n - 1
        for v in nums1:
            if v > t[i][0]:
                ans[t[i][1]] = v
                i += 1
            else:
                ans[t[j][1]] = v
                j -= 1
        return ans

sol = Solution()
for a, b, out, adv in examples:
    res = sol.advantageCount(list(a), list(b))
    ok = Counter(res) == Counter(a) and advantage(res, b) == adv
    print(res, "lợi thế", advantage(res, b), "OK" if ok else "SAI", "| trùng đáp án mẫu:", res == out)
    assert ok

[2, 11, 7, 15] lợi thế 4 OK | trùng đáp án mẫu: True
[24, 32, 8, 12] lợi thế 3 OK | trùng đáp án mẫu: True


## 10. Vì sao đúng

**Quy ước.** Giá trị trùng nhau không thắng nhau (điều kiện `>` nghiêm ngặt). Đặt `OPT(A, B)` là lợi thế tối đa khi ghép tập ngựa `A` với tập đối thủ `B` (bất kỳ cách ghép nào).

**Bổ đề (hoán đổi, exchange argument).** Cho `a` là con ngựa **yếu nhất** của ta và `b` là đối thủ **yếu nhất**.

1. **Nếu `a > b`:** tồn tại cách ghép tối ưu mà `a` đấu `b`. Lấy một cách ghép tối ưu bất kỳ, giả sử trong đó `a` đấu `b'` và `b` đấu `a'` (nếu đã là `a` đấu `b` thì xong). Ta có `a <= a'` và `b <= b'`. **Hoán đổi** thành `a` đấu `b` và `a'` đấu `b'`.
   - Trước khi đổi: số trận thắng của hai trận này là `[a > b'] + [a' > b]`.
   - Sau khi đổi: `1 + [a' > b']` (vì `a > b`).
   - Nếu `a > b'` thì `a' >= a > b'` nên `[a' > b'] = 1`: sau = 2 >= trước. Nếu không thì trước <= 1 <= sau.
   - Vậy không tệ hơn; do đó `OPT(A, B) = 1 + OPT(A \ {a}, B \ {b})`.
2. **Nếu `a <= b`:** `a` yếu nhất mà không thắng nổi đối thủ yếu nhất, nên không thắng được **bất kỳ** đối thủ nào (mọi đối thủ `>= b >= a`). Gọi `M` là đối thủ **mạnh nhất**. Lấy cách ghép tối ưu bất kỳ, giả sử `a` đấu `b''` và `M` đấu `a''` (nếu đã là `a` đấu `M` thì xong). Hoán đổi thành `a` đấu `M` và `a''` đấu `b''`.
   - Trước khi đổi: `a` thua, `[a'' > M]` là số trận thắng. Sau khi đổi: `a` vẫn thua, còn `[a'' > b'']`.
   - Vì `b'' <= M`, nên `[a'' > M] <= [a'' > b'']`: không tệ hơn.
   - Vậy `OPT(A, B) = OPT(A \ {a}, B \ {M})`.

Hai trường hợp trên chính là hai nhánh `if/else` của Cách 4. Quy nạp theo `n` cho kết quả tối ưu.

**Cách 2 và 3** là những cách duyệt khác nhưng cùng nguyên lý: (Cách 3) với đối thủ mạnh nhất `y`, nếu ngay con mạnh nhất cũng không thắng nổi thì đối thủ đó chắc chắn thắng mọi con của ta, hy sinh con yếu nhất; nếu con mạnh nhất thắng được thì dùng nó (nó thắng mọi đối thủ còn lại, và dùng cho đối thủ mạnh nhất là phù hợp theo hoán đổi đối xứng). (Cách 2) tham lam "con nhỏ nhất thắng được" cho từng đối thủ theo **thứ tự bất kỳ**: đúng vì tập đối thủ mà một con ngựa thắng được là một **đoạn đầu** của thứ tự sắp xếp (cấu trúc lồng nhau), nên ghép bằng "nhỏ nhất đủ sức" không lấy mất con ngựa cần cho đối thủ khác; chúng ta xác nhận bằng oracle.

**Đặc trưng độc lập cho oracle cỡ lớn.** `k` lượt thắng khả thi ⇔ `k` ngựa **mạnh nhất** của ta thắng được `k` đối thủ **yếu nhất** khi ghép **theo thứ tự đã sắp xếp** (`a_sorted[n-k+t] > b_sorted[t]` với mọi `t`). Cách này hoàn toàn độc lập với các lời giải và dùng tìm kiếm nhị phân `k` (tính đơn điệu vì nếu `k` khả thi thì `k-1` khả thi).

**Kiểm chứng bằng máy.** Dựng ba oracle: (1) vét cạn mọi hoán vị (`n <= 7`), (2) ghép cặp cực đại bằng thuật toán đường tăng Kuhn, (3) đặc trưng "k mạnh nhất vs k yếu nhất". Rồi kiểm tra: kết quả là hoán vị của `nums1` và lợi thế **bằng giá trị tối ưu** của oracle.

In [17]:
def opt_brute(nums1: List[int], nums2: List[int]) -> int:
    return max(advantage(p, nums2) for p in set(permutations(nums1)))

def opt_matching(nums1: List[int], nums2: List[int]) -> int:
    """Ghép cặp cực đại (Kuhn): cạnh (i, j) nếu nums1[i] > nums2[j]."""
    n = len(nums1)
    match_b = [-1] * n

    def try_aug(i: int, seen: List[bool]) -> bool:
        for j in range(n):
            if nums1[i] > nums2[j] and not seen[j]:
                seen[j] = True
                if match_b[j] == -1 or try_aug(match_b[j], seen):
                    match_b[j] = i
                    return True
        return False

    return sum(try_aug(i, [False] * n) for i in range(n))

def feasible(a_sorted: List[int], b_sorted: List[int], k: int) -> bool:
    n = len(a_sorted)
    return all(a_sorted[n - k + t] > b_sorted[t] for t in range(k))

def opt_characterization(nums1: List[int], nums2: List[int]) -> int:
    a, b = sorted(nums1), sorted(nums2)
    lo, hi = 0, len(a)
    while lo < hi:                    # k lớn nhất khả thi
        mid = (lo + hi + 1) // 2
        if feasible(a, b, mid):
            lo = mid
        else:
            hi = mid - 1
    return lo

def check_valid(nums1: List[int], nums2: List[int], res: List[int], opt: int) -> bool:
    return (len(res) == len(nums1)
            and Counter(res) == Counter(nums1)
            and advantage(res, nums2) == opt)

# ba oracle khớp nhau
rng = random.Random(1870)
for _ in range(400):
    n = rng.randint(1, 7)
    hi = rng.choice([3, 6, 20])
    a = [rng.randint(0, hi) for _ in range(n)]
    b = [rng.randint(0, hi) for _ in range(n)]
    o1, o2, o3 = opt_brute(a, b), opt_matching(a, b), opt_characterization(a, b)
    assert o1 == o2 == o3, (a, b, o1, o2, o3)
print("ba oracle (vét cạn / Kuhn / đặc trưng) khớp nhau trên 400 cặp ngẫu nhiên")

ba oracle (vét cạn / Kuhn / đặc trưng) khớp nhau trên 400 cặp ngẫu nhiên


## 11. Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Mọi hoán vị | `O(n! * n)` | `O(n)` |
| 2. `bisect` + danh sách sắp xếp | `O(n log n + n^2)` tệ nhất (do `pop(k)`), hằng số nhỏ | `O(n)` |
| 3. Điền Kỵ với `deque` | `O(n log n)` | `O(n)` |
| 4. Hai con trỏ (nộp) | `O(n log n)` | `O(n)` |

Đo thử:

In [18]:
rng = random.Random(11)
n = 100000
a = [rng.randint(0, 10 ** 9) for _ in range(n)]
b = [rng.randint(0, 10 ** 9) for _ in range(n)]
opt = opt_characterization(a, b)
for name, fn, size in (("Cách 4 hai con trỏ", Solution().advantageCount, n),
                        ("Cách 3 deque", sol_deque, n),
                        ("Cách 2 bisect", sol_bisect, n)):
    t0 = time.perf_counter()
    res = fn(list(a[:size]), list(b[:size]))
    dt = (time.perf_counter() - t0) * 1000
    assert check_valid(a[:size], b[:size], res, opt), name
    print(f"{name:20} n={size}: {dt:9.1f} ms, lợi thế {advantage(res, b[:size])} (tối ưu {opt})")

Cách 4 hai con trỏ   n=100000:     117.6 ms, lợi thế 99647 (tối ưu 99647)
Cách 3 deque         n=100000:      92.6 ms, lợi thế 99647 (tối ưu 99647)


Cách 2 bisect        n=100000:     502.4 ms, lợi thế 99647 (tối ưu 99647)


## 12. Bộ test

Gồm: ví dụ của đề, edge case, phản ví dụ cho các cách sai (tìm bằng tìm kiếm vét cạn), đối chiếu mọi cách với ba oracle, và test ngẫu nhiên (seed cố định, đúng ràng buộc). Vì nhiều đáp án đều đúng nên mọi kiểm tra dùng **hàm kiểm tra hợp lệ** `check_valid` (hoán vị của `nums1` và lợi thế bằng tối ưu).

In [19]:
solutions = {
    "brute": advantage_brute,
    "bisect": advantage_bisect,
    "deque": advantage_deque,
    "two": advantage_two_pointers,
    "sol_bisect": sol_bisect,
    "sol_deque": sol_deque,
    "sol_two": sol_two,
    "final": Solution().advantageCount,
}

def run_all(a, b, opt=None, skip_brute=False):
    if opt is None:
        opt = opt_characterization(a, b)
    for name, fn in solutions.items():
        if name == "brute" and (skip_brute or len(a) > 7):
            continue
        res = fn(list(a), list(b))
        assert check_valid(a, b, res, opt), (name, a, b, res, opt)
    return opt

# 12.1 ví dụ đề
for a, b, out, adv in examples:
    assert run_all(a, b) == adv
print("12.1 ví dụ đề: OK")

# 12.2 edge case (kỳ vọng tối ưu viết tay, kèm lý do)
edge = [
    ([5], [3], 1),                         # n = 1 thắng
    ([3], [3], 0),                         # bằng nhau không thắng
    ([2], [9], 0),                         # n = 1 thua
    ([0], [0], 0),                         # biên 0
    ([10 ** 9], [10 ** 9 - 1], 1),         # biên lớn
    ([1, 1, 1], [1, 1, 1], 0),             # toàn bằng nhau
    ([2, 2, 2], [1, 1, 1], 3),             # toàn thắng
    ([1, 1, 1], [2, 2, 2], 0),             # toàn thua
    ([1, 2, 3], [1, 2, 3], 2),             # 2>1, 3>2
    ([3, 2, 1], [1, 2, 3], 2),             # đã đảo thứ tự
    ([1, 2, 3, 4], [4, 3, 2, 1], 3),       # 2>1, 3>2, 4>3
    ([4, 4, 4, 4], [3, 3, 5, 5], 2),       # chỉ thắng được hai đối thủ 3
    ([1, 5, 9], [4, 4, 8], 2),
    ([0, 0, 10 ** 9, 10 ** 9], [0, 10 ** 9, 0, 10 ** 9], 2),   # 1e9 thắng 0 hai lần
]
for a, b, expect in edge:
    assert opt_matching(a, b) == expect, ("kỳ vọng sai", a, b, expect, opt_matching(a, b))
    assert run_all(a, b, expect) == expect
print("12.2 edge case: OK", len(edge), "ca")

12.1 ví dụ đề: OK
12.2 edge case: OK 14 ca


In [20]:
# 12.3 phản ví dụ: các cách sai (tìm bằng vét cạn nhỏ, rồi xác nhận lời giải đúng không bị lỗi)
def wrong_sorted_only(nums1, nums2):          # trả về nums1 đã sắp xếp (bỏ qua vị trí của nums2)
    return sorted(nums1)

def wrong_always_largest(nums1, nums2):       # luôn dùng con mạnh nhất nếu thắng, nếu không con yếu nhất; duyệt theo thứ tự gốc
    s = sorted(nums1)
    res = []
    for y in nums2:
        res.append(s.pop() if s[-1] > y else s.pop(0))
    return res

def wrong_ge(nums1, nums2):                   # hai con trỏ nhưng so sánh >= (ties tính là thắng)
    a = sorted(nums1)
    t = sorted((v, i) for i, v in enumerate(nums2))
    ans = [0] * len(a)
    i, j = 0, len(a) - 1
    for v in a:
        if v >= t[i][0]:
            ans[t[i][1]] = v
            i += 1
        else:
            ans[t[j][1]] = v
            j -= 1
    return ans

def wrong_index(nums1, nums2):                # tìm chỉ số bằng nums2.index(v): hỏng khi trùng giá trị
    a = sorted(nums1)
    sorted_b = sorted(nums2)
    ans = [0] * len(a)
    i, j = 0, len(a) - 1
    for v in a:
        if v > sorted_b[i]:
            ans[nums2.index(sorted_b[i])] = v
            i += 1
        else:
            ans[nums2.index(sorted_b[j])] = v
            j -= 1
    return ans

def find_counter(wrong, max_n=4, vals=4):
    from itertools import product
    for n in range(1, max_n + 1):
        for a in product(range(vals), repeat=n):
            for b in product(range(vals), repeat=n):
                try:
                    res = wrong(list(a), list(b))
                except Exception:
                    return list(a), list(b)
                if not check_valid(list(a), list(b), res, opt_brute(list(a), list(b))):
                    return list(a), list(b)
    return None

counters = {}
for name, fn in (("chỉ sắp xếp nums1", wrong_sorted_only), ("luôn dùng ngựa mạnh nhất (thứ tự gốc)", wrong_always_largest),
                 ("so sánh >=", wrong_ge), ("nums2.index khi trùng giá trị", wrong_index)):
    c = find_counter(fn)
    assert c is not None, name
    counters[name] = c
    a, b = c
    res = fn(list(a), list(b))
    opt = opt_brute(a, b)
    assert not check_valid(a, b, res, opt)
    print(f"{name:40} nums1={a} nums2={b}: sai -> {res} (lợi thế {advantage(res, b) if Counter(res) == Counter(a) else 'không là hoán vị'}, tối ưu {opt})")
    assert run_all(a, b, opt) == opt      # lời giải đúng không bị lỗi trên chính ca này
print("12.3 phản ví dụ: OK")

chỉ sắp xếp nums1                        nums1=[0, 1] nums2=[0, 1]: sai -> [0, 1] (lợi thế 0, tối ưu 1)
luôn dùng ngựa mạnh nhất (thứ tự gốc)    nums1=[1, 2] nums2=[0, 1]: sai -> [2, 1] (lợi thế 1, tối ưu 2)
so sánh >=                               nums1=[0, 1] nums2=[0, 1]: sai -> [0, 1] (lợi thế 0, tối ưu 1)
nums2.index khi trùng giá trị            nums1=[1, 1] nums2=[0, 0]: sai -> [1, 0] (lợi thế không là hoán vị, tối ưu 2)
12.3 phản ví dụ: OK


In [21]:
# 12.4 ngẫu nhiên nhỏ: so mọi lời giải với oracle vét cạn và Kuhn (seed cố định)
rng = random.Random(4870)
for _ in range(1500):
    n = rng.randint(1, 7)
    hi = rng.choice([2, 4, 10, 10 ** 9])
    a = [rng.randint(0, hi) for _ in range(n)]
    b = [rng.randint(0, hi) for _ in range(n)]
    opt = opt_brute(a, b)
    assert opt == opt_matching(a, b) == opt_characterization(a, b)
    run_all(a, b, opt)
print("12.4 ngẫu nhiên n <= 7 (so với vét cạn): OK")

# 12.5 trung bình: n tới 60, so với Kuhn và đặc trưng
rng = random.Random(5870)
for _ in range(300):
    n = rng.randint(8, 60)
    hi = rng.choice([3, 10, 100, 10 ** 9])
    a = [rng.randint(0, hi) for _ in range(n)]
    b = [rng.randint(0, hi) for _ in range(n)]
    opt = opt_matching(a, b)
    assert opt == opt_characterization(a, b)
    run_all(a, b, opt, skip_brute=True)
print("12.5 ngẫu nhiên n tới 60 (so với Kuhn): OK")

# 12.6 lớn: n = 10^5 so với đặc trưng độc lập (nhiều dạng dữ liệu)
rng = random.Random(6870)
n = 100000
shapes = {
    "ngẫu nhiên": ([rng.randint(0, 10 ** 9) for _ in range(n)], [rng.randint(0, 10 ** 9) for _ in range(n)]),
    "toàn trùng": ([7] * n, [7] * n),
    "giá trị ít": ([rng.randint(0, 3) for _ in range(n)], [rng.randint(0, 3) for _ in range(n)]),
    "tăng dần": (list(range(n)), list(range(n))),
    "nums1 hơn hẳn": (list(range(n, 2 * n)), list(range(n))),
}
for name, (a, b) in shapes.items():
    opt = opt_characterization(a, b)
    for fname in ("sol_deque", "sol_two", "final"):
        res = solutions[fname](list(a), list(b))
        assert check_valid(a, b, res, opt), (name, fname)
    print(f"  {name:14} n={n}: tối ưu = {opt}")
print("12.6 n = 10^5: OK")

# 12.7 tính chất: kết quả không làm đổi nums1 gốc khi ta truyền bản sao, và lợi thế bất biến khi xáo trộn nums1
rng = random.Random(7870)
for _ in range(300):
    n = rng.randint(1, 30)
    a = [rng.randint(0, 10) for _ in range(n)]
    b = [rng.randint(0, 10) for _ in range(n)]
    shuffled = a[:]
    rng.shuffle(shuffled)
    r1 = Solution().advantageCount(a[:], b[:])
    r2 = Solution().advantageCount(shuffled, b[:])
    assert advantage(r1, b) == advantage(r2, b)
print("12.7 lợi thế không phụ thuộc thứ tự ban đầu của nums1: OK")

12.4 ngẫu nhiên n <= 7 (so với vét cạn): OK


12.5 ngẫu nhiên n tới 60 (so với Kuhn): OK


  ngẫu nhiên     n=100000: tối ưu = 99729
  toàn trùng     n=100000: tối ưu = 0


  giá trị ít     n=100000: tối ưu = 74728


  tăng dần       n=100000: tối ưu = 99999


  nums1 hơn hẳn  n=100000: tối ưu = 100000
12.6 n = 10^5: OK
12.7 lợi thế không phụ thuộc thứ tự ban đầu của nums1: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ sắp xếp `nums1` rồi trả về

Sắp xếp `nums1` là cần thiết nhưng **chưa đủ**: kết quả phải xếp theo thứ tự của `nums2`.

In [22]:
a, b = counters["chỉ sắp xếp nums1"]
print("nums1 =", a, "nums2 =", b, "| sai ->", wrong_sorted_only(a, b), "lợi thế", advantage(wrong_sorted_only(a, b), b), "| đúng ->", Solution().advantageCount(list(a), list(b)), "lợi thế", advantage(Solution().advantageCount(list(a), list(b)), b))
assert advantage(wrong_sorted_only(a, b), b) < opt_brute(a, b)
assert advantage(Solution().advantageCount(list(a), list(b)), b) == opt_brute(a, b)

nums1 = [0, 1] nums2 = [0, 1] | sai -> [0, 1] lợi thế 0 | đúng -> [1, 0] lợi thế 1


Vì sao sai: kết quả là hoán vị đúng nhưng đặt các con ngựa sai **vị trí**. **Đúng:** sắp xếp cả `nums2` kèm chỉ số gốc (`(giá trị, chỉ số)`) để đặt đúng vị trí.

### Lỗi 2 — Dùng `nums2.index(v)` để tìm vị trí

Khi `nums2` có giá trị **trùng**, `index` luôn trả vị trí xuất hiện **đầu tiên**: nhiều con ngựa ghi đè lên cùng một ô, các ô khác bị bỏ trống (còn `0`).

In [23]:
a, b = counters["nums2.index khi trùng giá trị"]
print("nums1 =", a, "nums2 =", b, "| sai ->", wrong_index(a, b), "| đúng ->", Solution().advantageCount(list(a), list(b)))
assert Counter(wrong_index(a, b)) != Counter(a) or advantage(wrong_index(a, b), b) < opt_brute(a, b)

nums1 = [1, 1] nums2 = [0, 0] | sai -> [1, 0] | đúng -> [1, 1]


Vì sao sai: hai đối thủ cùng giá trị cùng nhận cùng một chỉ số. **Đúng:** sắp xếp danh sách `(giá trị, chỉ số)` (mỗi đối thủ có chỉ số riêng), hoặc `sorted(range(n), key=...)`.

### Lỗi 3 — So sánh `>=` thay vì `>`

Đề đếm chỉ số `nums1[i] > nums2[i]` (nghiêm ngặt). Với `>=`, hai giá trị bằng nhau bị tính như một chiến thắng và thuật toán dùng nhầm ngựa.

In [24]:
a, b = counters["so sánh >="]
res_wrong, res_right = wrong_ge(a, b), Solution().advantageCount(list(a), list(b))
print("nums1 =", a, "nums2 =", b, "| sai ->", res_wrong, "lợi thế", advantage(res_wrong, b), "| đúng ->", res_right, "lợi thế", advantage(res_right, b))
assert advantage(res_wrong, b) < advantage(res_right, b) == opt_brute(a, b)

nums1 = [0, 1] nums2 = [0, 1] | sai -> [0, 1] lợi thế 0 | đúng -> [1, 0] lợi thế 1


Vì sao sai: ví dụ `nums1 = [x]`, `nums2 = [x]` cho lợi thế `0`, không phải `1`. **Đúng:** dùng `>` ở mọi chỗ (cả trong `bisect_right`, vốn đã tìm phần tử `> y`).

### Lỗi 4 — Tham lam "luôn dùng con mạnh nhất" theo thứ tự gốc

Nghe hợp lý ("có con mạnh thì cứ dùng") nhưng **phí** ngựa mạnh cho đối thủ yếu.

In [25]:
a, b = counters["luôn dùng ngựa mạnh nhất (thứ tự gốc)"]
res_wrong, res_right = wrong_always_largest(a, b), Solution().advantageCount(list(a), list(b))
print("nums1 =", a, "nums2 =", b, "| sai ->", res_wrong, "lợi thế", advantage(res_wrong, b), "| đúng ->", res_right, "lợi thế", advantage(res_right, b))
assert advantage(res_wrong, b) < advantage(res_right, b) == opt_brute(a, b)

nums1 = [1, 2] nums2 = [0, 1] | sai -> [2, 1] lợi thế 1 | đúng -> [1, 2] lợi thế 2


Vì sao sai: dùng con mạnh nhất cho đối thủ yếu nhất làm mất con mạnh nhất, khiến một đối thủ khác thắng được. **Đúng:** dùng con **nhỏ nhất đủ sức thắng** (Cách 2) hoặc xử lý từ đối thủ mạnh nhất (Cách 3).

### Lỗi 5 — Bỏ qua nhánh "hy sinh"

Nếu con ngựa hiện tại không thắng được đối thủ yếu nhất, bỏ qua (không gán) nó làm một ô trong `ans` giữ giá trị mặc định `0`.

In [26]:
def wrong_no_sacrifice(nums1, nums2):
    a = sorted(nums1)
    t = sorted((v, i) for i, v in enumerate(nums2))
    ans = [0] * len(a)
    i = 0
    for v in a:
        if v > t[i][0]:
            ans[t[i][1]] = v
            i += 1
    return ans

a, b = [1, 1], [5, 5]
res = wrong_no_sacrifice(a, b)
print("nums1 =", a, "nums2 =", b, "| sai ->", res, "| đúng ->", Solution().advantageCount(list(a), list(b)))
assert Counter(res) != Counter(a)

nums1 = [1, 1] nums2 = [5, 5] | sai -> [0, 0] | đúng -> [1, 1]


Vì sao sai: kết quả không còn là hoán vị của `nums1` (xuất hiện số `0` không có trong `nums1`). **Đúng:** luôn gán mỗi con ngựa một vị trí: nếu không thắng thì cho đối thủ mạnh nhất còn lại.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** ghép cặp hai tập để đếm số cặp thỏa `x > y` nghiêm ngặt; "trả về một hoán vị tối ưu" ⇒ **tham lam sau khi sắp xếp** (cuộc đua ngựa Điền Kỵ).
- **Hai quy tắc vàng:** (1) con nhỏ nhất đủ sức thắng; (2) con vô dụng thì hy sinh cho đối thủ mạnh nhất.
- **Nhớ vị trí gốc:** sắp xếp `(giá trị, chỉ số)` hoặc `sorted(range(n), key=...)`.
- Khi đề cho **nhiều đáp án**: viết hàm kiểm tra `là hoán vị` + `giá trị mục tiêu bằng tối ưu`, đừng so từng phần tử.
- `bisect_right` cho `>`, `bisect_left` cho `>=`.
- Cùng họ: Assign Cookies (455), Boats to Save People (881), Two City Scheduling (1029), Maximum Matching of Players With Trainers (2410), Delete and Earn (740).
- Test nên có: `n = 1`, mọi giá trị bằng nhau, giá trị `0` và `10^9`, trùng nhiều, `nums1` hơn hẳn `nums2` và ngược lại.

## 15. Biến thể

1. **Chỉ cần số lượt thắng tối đa** (không cần hoán vị): `O(n log n)` bằng hai con trỏ ghép "con nhỏ nhất thắng được".
2. **Điểm: thắng `+1`, hòa `0`, thua `-1`** (Điền Kỵ gốc): cần xét thêm trường hợp hòa (bài tập 5).
3. **Lợi thế không nghiêm ngặt** (`>=`): đổi `bisect_right` thành `bisect_left`.
4. Bài liên quan: Maximum Matching of Players With Trainers (2410), Assign Cookies (455).

Dưới đây là code biến thể 1 kèm kiểm chứng bằng ba oracle.

In [27]:
def max_advantage_count(nums1: List[int], nums2: List[int]) -> int:
    a, b = sorted(nums1), sorted(nums2)
    j = 0
    for v in a:
        if v > b[j]:
            j += 1
    return j

rng = random.Random(15)
for _ in range(500):
    n = rng.randint(1, 8)
    a = [rng.randint(0, 6) for _ in range(n)]
    b = [rng.randint(0, 6) for _ in range(n)]
    assert max_advantage_count(a, b) == opt_matching(a, b) == opt_characterization(a, b)
    if n <= 7:
        assert max_advantage_count(a, b) == opt_brute(a, b)
print("max_advantage_count khớp mọi oracle; ví dụ:", max_advantage_count([2, 7, 11, 15], [1, 10, 4, 11]), max_advantage_count([12, 24, 8, 32], [13, 25, 32, 11]))

max_advantage_count khớp mọi oracle; ví dụ: 4 3


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi hoán vị | Định nghĩa | `O(n! n)` | `O(n)` | Oracle, `n <= 8` |
| 2. `bisect` + danh sách | Nhỏ nhất đủ sức, theo thứ tự gốc | `O(n^2)` tệ nhất | `O(n)` | Dễ hiểu; `n` vừa |
| 3. Điền Kỵ + `deque` | Mạnh nhất trước | `O(n log n)` | `O(n)` | Dạng cổ điển, dễ chứng minh |
| 4. Hai con trỏ | Yếu nhất trước | `O(n log n)` | `O(n)` | **Cách nộp chính** |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán output ba dòng, rồi chạy để kiểm tra:

```python
print(advantage([2, 3], [1, 3]))
print(sorted((v, i) for i, v in enumerate([5, 1, 5])))
print(bisect_right([1, 3, 3, 7], 3))
```

In [28]:
# Bài 1: ghi dự đoán của bạn
my_guess_line1 = None
my_guess_line2 = None
my_guess_line3 = None

### Bài 2 — Tính tay (dễ-vừa)

Với `nums1 = [1, 5, 9]`, `nums2 = [4, 4, 8]`: ghi lợi thế tối đa và **một** hoán vị đạt lợi thế đó.

In [29]:
# Bài 2: điền kết quả tính tay
hand_best_advantage = None
hand_permutation = None

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành hai con trỏ trên hai mảng đã sắp xếp.

In [30]:
def exercise_3_two_pointers(nums1: List[int], nums2: List[int]) -> List[int]:
    a = sorted(nums1)
    t = sorted((v, i) for i, v in enumerate(nums2))
    n = len(nums2)
    ans = [0] * n
    i, j = 0, n - 1
    for v in a:
        # TODO 1: nếu v thắng được đối thủ yếu nhất còn lại t[i]: gán và tăng i
        # TODO 2: ngược lại: hy sinh cho đối thủ mạnh nhất t[j] và giảm j
        pass
    return ans


# for a, b, out, adv in examples:
#     r = exercise_3_two_pointers(list(a), list(b))
#     assert Counter(r) == Counter(a) and advantage(r, b) == adv

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 2, hãy tự viết `exercise_4_bisect(nums1, nums2)` dùng danh sách đã sắp xếp và `bisect_right` (nhỏ nhất đủ sức thắng, không có thì lấy con nhỏ nhất).

In [31]:
def exercise_4_bisect(nums1: List[int], nums2: List[int]) -> List[int]:
    # TODO
    pass


# for a, b, out, adv in examples:
#     r = exercise_4_bisect(list(a), list(b))
#     assert Counter(r) == Counter(a) and advantage(r, b) == adv

### Bài 5 — Mini challenge: cuộc đua Điền Kỵ có hòa (khó)

Cuộc đua ngựa: thắng `+1`, **hòa `0`** (bằng nhau), thua `-1`. Hãy viết `exercise_5_tianji(a, b)` trả về **điểm tối đa** có thể đạt được khi ghép tùy ý.

```python
exercise_5_tianji([4, 5, 6], [3, 4, 5])  -> 3     # 4>3, 5>4, 6>5
exercise_5_tianji([1, 1], [1, 1])        -> 0     # hòa cả hai
exercise_5_tianji([1], [2])              -> -1
exercise_5_tianji([2, 2, 2], [3, 3, 3])  -> -3
```
Gợi ý: sắp xếp cả hai; so ngựa mạnh nhất của ta với mạnh nhất của đối thủ; nếu thắng ngay thì thắng; nếu không thì so ngựa yếu nhất với yếu nhất; nếu còn không thắng thì cho ngựa yếu nhất đấu với đối thủ mạnh nhất.

In [32]:
def exercise_5_tianji(a: List[int], b: List[int]) -> int:
    # TODO
    pass


# assert exercise_5_tianji([4, 5, 6], [3, 4, 5]) == 3
# assert exercise_5_tianji([1, 1], [1, 1]) == 0
# assert exercise_5_tianji([1], [2]) == -1
# assert exercise_5_tianji([2, 2, 2], [3, 3, 3]) == -3

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `advantage` đếm số chỉ số `a[i] > b[i]`.
- Mức 2: `[2, 3]` vs `[1, 3]`: `2 > 1` đúng, `3 > 3` sai. `bisect_right` trả vị trí đầu tiên có phần tử `> x`.
- Mức 3: `1`, `[(1, 1), (5, 0), (5, 2)]`, `3`.

**Bài 2**
- Mức 1: đối thủ `4, 4, 8`; ngựa của ta `1, 5, 9`.
- Mức 2: `5 > 4` và `9 > 8`; `1` chẳng thắng ai.
- Mức 3: lợi thế `2`; ví dụ `[5, 1, 9]` (`5>4`, `1<4`, `9>8`). Có đáp án khác như `[1, 5, 9]` hay không? Hãy kiểm tra.

**Bài 3**
- Mức 1: mỗi vòng lặp xử lý đúng một con `v`.
- Mức 2: `if v > t[i][0]: ans[t[i][1]] = v; i += 1`.
- Mức 3: `else: ans[t[j][1]] = v; j -= 1`.

**Bài 4**
- Mức 1: `s = sorted(nums1)`; với mỗi `y` trong `nums2` (thứ tự gốc).
- Mức 2: `k = bisect_right(s, y)`.
- Mức 3: `res.append(s.pop(k if k < len(s) else 0))`.

**Bài 5**
- Mức 1: bốn con trỏ: `al, ar` cho ta, `bl, br` cho đối thủ.
- Mức 2: nếu `a[ar] > b[br]`: thắng; elif `a[al] > b[bl]`: thắng; else so `a[al]` với `b[br]`.
- Mức 3: trong nhánh else, điểm cộng `(a[al] > b[br]) - (a[al] < b[br])`, rồi `al += 1`, `br -= 1`.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [33]:
print(advantage([2, 3], [1, 3]))
print(sorted((v, i) for i, v in enumerate([5, 1, 5])))
print(bisect_right([1, 3, 3, 7], 3))

assert advantage([2, 3], [1, 3]) == 1
assert sorted((v, i) for i, v in enumerate([5, 1, 5])) == [(1, 1), (5, 0), (5, 2)]
assert bisect_right([1, 3, 3, 7], 3) == 3

1
[(1, 1), (5, 0), (5, 2)]
3


### Đáp án Bài 2

Lợi thế tối đa `2` (ví dụ `[5, 1, 9]`).

In [34]:
hand_best_advantage_solution = 2
hand_permutation_solution = [5, 1, 9]
nums1, nums2 = [1, 5, 9], [4, 4, 8]
assert opt_brute(nums1, nums2) == hand_best_advantage_solution
assert check_valid(nums1, nums2, hand_permutation_solution, hand_best_advantage_solution)
assert check_valid(nums1, nums2, Solution().advantageCount(nums1[:], nums2[:]), 2)
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [35]:
def exercise_3_two_pointers_solution(nums1: List[int], nums2: List[int]) -> List[int]:
    a = sorted(nums1)
    t = sorted((v, i) for i, v in enumerate(nums2))
    n = len(nums2)
    ans = [0] * n
    i, j = 0, n - 1
    for v in a:
        if v > t[i][0]:
            ans[t[i][1]] = v
            i += 1
        else:
            ans[t[j][1]] = v
            j -= 1
    return ans

for a, b, out, adv in examples:
    r = exercise_3_two_pointers_solution(list(a), list(b))
    assert Counter(r) == Counter(a) and advantage(r, b) == adv
rng = random.Random(3)
for _ in range(300):
    n = rng.randint(1, 8)
    a = [rng.randint(0, 6) for _ in range(n)]
    b = [rng.randint(0, 6) for _ in range(n)]
    assert check_valid(a, b, exercise_3_two_pointers_solution(a[:], b[:]), opt_matching(a, b))
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [36]:
def exercise_4_bisect_solution(nums1: List[int], nums2: List[int]) -> List[int]:
    s = sorted(nums1)
    res = []
    for y in nums2:
        k = bisect_right(s, y)
        res.append(s.pop(k if k < len(s) else 0))
    return res

for a, b, out, adv in examples:
    r = exercise_4_bisect_solution(list(a), list(b))
    assert Counter(r) == Counter(a) and advantage(r, b) == adv
rng = random.Random(4)
for _ in range(300):
    n = rng.randint(1, 8)
    a = [rng.randint(0, 6) for _ in range(n)]
    b = [rng.randint(0, 6) for _ in range(n)]
    assert check_valid(a, b, exercise_4_bisect_solution(a[:], b[:]), opt_matching(a, b))
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [37]:
def exercise_5_tianji_solution(a: List[int], b: List[int]) -> int:
    a, b = sorted(a), sorted(b)
    al, ar, bl, br = 0, len(a) - 1, 0, len(b) - 1
    score = 0
    while al <= ar:
        if a[ar] > b[br]:
            score += 1
            ar -= 1
            br -= 1
        elif a[al] > b[bl]:
            score += 1
            al += 1
            bl += 1
        else:
            score += (a[al] > b[br]) - (a[al] < b[br])
            al += 1
            br -= 1
    return score

def tianji_brute(a: List[int], b: List[int]) -> int:
    return max(sum((x > y) - (x < y) for x, y in zip(p, b)) for p in permutations(a))

assert exercise_5_tianji_solution([4, 5, 6], [3, 4, 5]) == 3
assert exercise_5_tianji_solution([1, 1], [1, 1]) == 0
assert exercise_5_tianji_solution([1], [2]) == -1
assert exercise_5_tianji_solution([2, 2, 2], [3, 3, 3]) == -3
rng = random.Random(5)
for _ in range(1500):
    n = rng.randint(1, 7)
    a = [rng.randint(0, 6) for _ in range(n)]
    b = [rng.randint(0, 6) for _ in range(n)]
    assert exercise_5_tianji_solution(a, b) == tianji_brute(a, b), (a, b)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (hai con trỏ trên mảng đã sắp xếp)

```python
class Solution:
    def advantageCount(self, nums1: List[int], nums2: List[int]) -> List[int]:
        nums1.sort()
        t = sorted((v, i) for i, v in enumerate(nums2))   # đối thủ kèm vị trí gốc
        n = len(nums2)
        ans = [0] * n
        i, j = 0, n - 1                                    # yếu nhất / mạnh nhất còn lại
        for v in nums1:                                    # ngựa của ta từ yếu đến mạnh
            if v > t[i][0]:                                # thắng được đối thủ yếu nhất -> ghép
                ans[t[i][1]] = v
                i += 1
            else:                                          # vô dụng -> hy sinh cho đối thủ mạnh nhất
                ans[t[j][1]] = v
                j -= 1
        return ans
```

Công thức nhớ nhanh: **nhỏ nhất đủ sức thắng**; vô dụng thì **cho đối thủ mạnh nhất**.

### Checklist trước khi submit

- [ ] Kết quả xếp theo **thứ tự của `nums2`** (dùng `(giá trị, chỉ số)`), không phải thứ tự sắp xếp?
- [ ] So sánh **`>` nghiêm ngặt**, không `>=`?
- [ ] Mỗi ngựa đều được gán (kể cả khi thua) để kết quả vẫn là hoán vị của `nums1`?
- [ ] Không dùng `nums2.index(...)` khi có giá trị trùng?
- [ ] Kiểm thử bằng **hàm kiểm tra hợp lệ** (hoán vị + lợi thế tối ưu), không so từng phần tử?
- [ ] Đã thử: `n = 1`, toàn bằng nhau, `0` và `10^9`, trùng nhiều, `n = 10^5`?
- [ ] Tên method đúng `advantageCount(self, nums1, nums2)` và trả `List[int]`?